# 05 · 类别变量分析：父母职业是否存在关联

**数据**：`E:\类别变量练习数据.csv`（GBK 编码，53 行 × 2 列，单位名称为自由文本）

**流程**：清洗 → 职业归类 → 列联表 → 卡方检验（+ 置换检验）→ 效应量 → 可视化

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei"]
plt.rcParams["axes.unicode_minus"] = False

df = pd.read_csv(r"E:\类别变量练习数据.csv", encoding="gbk")
df.columns = ["父亲工作", "母亲工作"]
print(df.shape)
df.head(8)

## 第一步：清洗

原始数据的问题：部分单元格内含换行符与首尾空格；母亲工作有 1 条缺失。

In [ ]:
df["父亲工作"] = df["父亲工作"].astype(str).str.replace("\n", "").str.strip()
df["母亲工作"] = df["母亲工作"].astype(str).str.replace("\n", "").str.strip().replace("nan", np.nan)

print("母亲工作缺失：", df["母亲工作"].isna().sum())
print("父亲工作唯一值：", df["父亲工作"].nunique(), "| 母亲工作唯一值：", df["母亲工作"].nunique())

## 第二步：把「单位名称」归类为「职业类别」

原始字段是公司/单位名，无法直接做统计。按关键词归类为 6 类：
党政机关 / 事业单位 / 国有企业 / 民营企业 / 个体经营 / 家庭主妇·无业。

> 注意：关键词规则基于单位名称，属于**启发式归类**，个别单位可能有争议（如“银光聚银化工”按国企处理）。

In [ ]:
gov = ["局", "法院", "纪委", "市委", "镇", "管委会", "街道", "公安", "监狱", "团委", "执法", "审计", "统战", "编办", "社区"]
soe = ["中铁", "中海油", "兵器工业", "水利水电", "二十一冶", "农行", "给排水", "银光聚银"]
inst = ["大学", "中学", "学校", "医院", "学院", "保健站", "中心", "广播电视", "管理处"]
indiv = ["个体", "店", "馆", "菜市场", "烟酒", "浴", "农资", "食府", "书店", "大市场", "汽修"]

def classify(x):
    if pd.isna(x):
        return "缺失"
    s = str(x)
    if any(k in s for k in ["家庭主妇", "家庭父亲", "家庭父母"]) or s == "无":
        return "家庭主妇/无业"
    if s == "个体" or any(k in s for k in indiv):
        return "个体经营"
    if any(k in s for k in soe):
        return "国有企业"
    if any(k in s for k in inst):
        return "事业单位"
    if any(k in s for k in gov):
        return "党政机关"
    if any(k in s for k in ["有限公司", "有限责任公司", "公司", "旅行社", "养殖", "工贸"]):
        return "民营企业"
    return "其他/未识别"

df["父亲职业类别"] = df["父亲工作"].map(classify)
df["母亲职业类别"] = df["母亲工作"].map(classify)

print("未识别条数：", ((df["父亲职业类别"] == "其他/未识别") | (df["母亲职业类别"] == "其他/未识别")).sum())
print(df["父亲职业类别"].value_counts().to_string())
print()
print(df["母亲职业类别"].value_counts().to_string())

## 第三步：合并为 4 类并构建列联表

原始 6 类在 52 个样本下过于稀疏（期望频数太小），因此合并为：**体制内**（党政+事业+国企）、**民营**、**个体**、**无业**。

In [ ]:
m = {"党政机关": "体制内", "事业单位": "体制内", "国有企业": "体制内",
     "民营企业": "民营", "个体经营": "个体", "家庭主妇/无业": "无业"}

d = df[(df["父亲职业类别"] != "缺失") & (df["母亲职业类别"] != "缺失")].copy()
d["父亲合并"] = d["父亲职业类别"].map(m)
d["母亲合并"] = d["母亲职业类别"].map(m)

order = ["体制内", "民营", "个体", "无业"]
tab = pd.crosstab(d["父亲合并"], d["母亲合并"]).reindex(index=order, columns=order, fill_value=0)
tab

## 第四步：卡方检验

H₀：父母职业类别相互独立（不存在关联）
H₁：两者存在关联

In [ ]:
chi2, p, dof, exp = chi2_contingency(tab)
n = tab.values.sum()
v = np.sqrt(chi2 / (n * (min(tab.shape) - 1)))   # Cramér's V

print(f"样本量 N = {n}")
print(f"卡方值 = {chi2:.3f}, 自由度 = {dof}, p = {p:.4f}")
print(f"Cramér's V = {v:.3f}")
print(f"最小期望频数 = {exp.min():.2f}；期望频数<5 的格子：{(exp<5).sum()}/{exp.size}")

print("\n期望频数表：")
print(pd.DataFrame(exp, index=tab.index, columns=tab.columns).round(2))

**注意**：16 个格子里有 13 个期望频数 < 5，严格来说不满足卡方检验的近似条件。
因此下面再用**置换检验（permutation test）**做一次不依赖大样本近似的验证。

In [ ]:
# 对角线：夫妻同属一类别的比例
diag = sum(tab.iloc[i, i] for i in range(4))
exp_diag = sum(exp[i, i] for i in range(4))
print(f"同类别夫妻：{diag}/{n} = {diag/n*100:.1f}%，随机期望 {exp_diag/n*100:.1f}%")

# 置换检验：打乱母亲职业后重算卡方
rng = np.random.default_rng(42)
cnt = 0
for _ in range(10000):
    perm = rng.permutation(d["母亲合并"].values)
    t = pd.crosstab(d["父亲合并"], perm).reindex(index=order, columns=order, fill_value=0)
    if chi2_contingency(t)[0] >= chi2:
        cnt += 1
print(f"置换检验 p ≈ {cnt/10000:.4f}")

## 第五步：可视化

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.4))

sns.heatmap(tab, annot=True, fmt="d", cmap="Blues", cbar=False,
            linewidths=.6, linecolor="white", ax=axes[0])
axes[0].set_title("父母职业类别列联表（人数）")
axes[0].set_xlabel("母亲职业类别"); axes[0].set_ylabel("父亲职业类别")

pct = tab.div(tab.sum(axis=1), axis=0) * 100
sns.heatmap(pct, annot=True, fmt=".0f", cmap="Oranges", cbar=False,
            linewidths=.6, linecolor="white", ax=axes[1], vmin=0, vmax=100)
axes[1].set_title("按父亲职业分行看分布（行百分比 %）")
axes[1].set_xlabel("母亲职业类别"); axes[1].set_ylabel("")

plt.tight_layout()
plt.savefig(r"D:\人工智能\analysis\父母职业关联_热力图.png", dpi=150, bbox_inches="tight")
plt.show()

## 结论

1. **存在统计显著关联**：卡方 p ≈ 0.004，置换检验 p ≈ 0.005，均 < 0.01。
2. **关联强度中等**：Cramér's V ≈ 0.39。
3. **主要来源**：体制内—体制内组合 21 对，远超随机期望；父亲民营且母亲无业 5 对，也高于期望。
4. **同类别比例**：55.8%，明显高于独立假设下的 34.4%。

### 局限（写报告必须写）
- 样本仅 52 对，且是**便利样本**（非随机抽样），不能推断总体。
- 职业类别由单位名称关键词推断，存在**归类误差**（如“银光聚银化工”是否国企）。
- 卡方期望频数偏小，故补做了置换检验；但小样本下效应量估计不稳定。
- 关联 ≠ 因果关系，数据也无法排除地域、教育、户籍等混杂因素。